# A02 – Data Source Setup
Kiểm tra 9 CSV Olist, profiling dữ liệu nguồn và tạo ngày lễ quốc gia Brazil 2016–2018. Notebook chạy từ một bản clone repository và không dùng Google Drive.

In [3]:
%cd /content
!git clone https://github.com/huyenpiano1711/CommerceFlow360.git
%cd /content/CommerceFlow360

/content
Cloning into 'CommerceFlow360'...
remote: Enumerating objects: 64, done.
remote: Counting objects: 100% (64/64), done.
remote: Compressing objects: 100% (49/49), done.
remote: Total 64 (delta 4), reused 50 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (64/64), 42.31 MiB | 14.30 MiB/s, done.
Resolving deltas: 100% (4/4), done.
Updating files: 100% (44/44), done.
/content/CommerceFlow360


In [4]:
from pathlib import Path
import sys

IN_COLAB = 'google.colab' in sys.modules
print('Google Colab:', IN_COLAB)

def find_project_root(start=Path.cwd()):
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / '.git').exists() and (candidate / 'data' / 'raw').is_dir():
            return candidate
    raise FileNotFoundError('Clone/open the CommerceFlow360 repository first.')

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT))
print('Project root:', PROJECT_ROOT)

Google Colab: True
Project root: /content/CommerceFlow360


## Cài dependencies
Bỏ comment và chạy cell dưới nếu môi trường chưa có dependencies. Trên Colab, lệnh đọc trực tiếp `requirements.txt` từ repository vừa clone.

In [5]:
# %pip install -q -r {PROJECT_ROOT / 'requirements.txt'}

In [6]:
OLIST_FILES = [
    'olist_customers_dataset.csv',
    'olist_geolocation_dataset.csv',
    'olist_order_items_dataset.csv',
    'olist_order_payments_dataset.csv',
    'olist_order_reviews_dataset.csv',
    'olist_orders_dataset.csv',
    'olist_products_dataset.csv',
    'olist_sellers_dataset.csv',
    'product_category_name_translation.csv',
]
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
available = [name for name in OLIST_FILES if (RAW_DIR / name).is_file()]
missing = [name for name in OLIST_FILES if name not in available]
print(f'Found {len(available)}/9 Olist CSV files')
print('Available:', available)
print('Missing:', missing)

Found 9/9 Olist CSV files
Available: ['olist_customers_dataset.csv', 'olist_geolocation_dataset.csv', 'olist_order_items_dataset.csv', 'olist_order_payments_dataset.csv', 'olist_order_reviews_dataset.csv', 'olist_orders_dataset.csv', 'olist_products_dataset.csv', 'olist_sellers_dataset.csv', 'product_category_name_translation.csv']
Missing: []


## Tùy chọn: tải riêng file còn thiếu từ Kaggle
Cell không chạy mặc định và không ghi đè file sẵn có. Nếu Kaggle yêu cầu xác thực, cấu hình credential bằng Colab Secrets/biến môi trường; không dán token vào notebook.

In [7]:
DOWNLOAD_MISSING = False
if DOWNLOAD_MISSING and missing:
    import shutil
    import kagglehub
    download_dir = Path(kagglehub.dataset_download('olistbr/brazilian-ecommerce'))
    for name in missing:
        candidates = list(download_dir.rglob(name))
        if not candidates:
            print('Not found in Kaggle download:', name)
            continue
        destination = RAW_DIR / name
        if not destination.exists():
            shutil.copy2(candidates[0], destination)
            print('Copied:', name)
elif missing:
    print('Set DOWNLOAD_MISSING=True to download only missing files.')

In [8]:
import pandas as pd
from IPython.display import display

inventory = []
for name in OLIST_FILES:
    path = RAW_DIR / name
    if path.is_file():
        header = pd.read_csv(path, nrows=0)
        inventory.append({'file_name': name, 'column_count': len(header.columns), 'size_mb': round(path.stat().st_size / 1024**2, 3)})
display(pd.DataFrame(inventory))

,file_name,column_count,size_mb
0,olist_customers_dataset.csv,5,8.615
1,olist_geolocation_dataset.csv,5,58.435
2,olist_order_items_dataset.csv,7,14.723
3,olist_order_payments_dataset.csv,5,5.510
4,olist_order_reviews_dataset.csv,7,13.682
5,olist_orders_dataset.csv,8,16.837
6,olist_products_dataset.csv,9,2.269
7,olist_sellers_dataset.csv,4,0.167
8,product_category_name_translation.csv,2,0.002


In [9]:
from src.validation.profile_raw_data import run as run_profiling
profile_result = run_profiling(PROJECT_ROOT)
display(profile_result)

,file_name,actual_rows,reference_rows,difference,column_count,size_mb,status,duplicate_rows,notes
0,olist_customers_dataset.csv,99441,<NA>,<NA>,5,8.615,PASS,0,Schema and CSV readability validated; referenc...
1,olist_geolocation_dataset.csv,1000163,<NA>,<NA>,5,58.435,PASS,261831,Schema and CSV readability validated; referenc...
2,olist_order_items_dataset.csv,112650,<NA>,<NA>,7,14.723,PASS,0,Schema and CSV readability validated; referenc...
3,olist_order_payments_dataset.csv,103886,<NA>,<NA>,5,5.510,PASS,0,Schema and CSV readability validated; referenc...
4,olist_order_reviews_dataset.csv,99224,<NA>,<NA>,7,13.682,PASS,0,Schema and CSV readability validated; referenc...
5,olist_orders_dataset.csv,99441,<NA>,<NA>,8,16.837,PASS,0,Schema and CSV readability validated; referenc...
6,olist_products_dataset.csv,32951,<NA>,<NA>,9,2.269,PASS,0,Schema and CSV readability validated; referenc...
7,olist_sellers_dataset.csv,3095,<NA>,<NA>,4,0.167,PASS,0,Schema and CSV readability validated; referenc...
8,product_category_name_translation.csv,71,<NA>,<NA>,2,0.002,PASS,0,Schema and CSV readability validated; referenc...


In [10]:
from src.ingestion.generate_brazil_holidays import generate
holiday_result = generate(PROJECT_ROOT)
display(holiday_result)

,date,holiday_name,country_code,is_holiday
0,2016-01-01,Universal Fraternization Day,BR,1
1,2016-03-25,Good Friday,BR,1
2,2016-04-21,Tiradentes' Day,BR,1
3,2016-05-01,Worker's Day,BR,1
4,2016-09-07,Independence Day,BR,1
5,2016-10-12,Our Lady of Aparecida,BR,1
6,2016-11-02,All Souls' Day,BR,1
7,2016-11-15,Republic Proclamation Day,BR,1
8,2016-12-25,Christmas Day,BR,1
9,2017-01-01,Universal Fraternization Day,BR,1


In [11]:
checks = {
    '9 Olist CSV files': all((RAW_DIR / name).is_file() for name in OLIST_FILES),
    'all Olist schemas pass': len(profile_result) == 9 and profile_result['status'].eq('PASS').all(),
    'comparison output exists': (PROJECT_ROOT / 'data/profiling/doi_chieu_so_dong.csv').is_file(),
    'holiday output exists': (RAW_DIR / 'holidays_brazil.csv').is_file(),
    'holiday dates unique': not holiday_result['date'].duplicated().any(),
    'holiday required fields complete': not holiday_result.isna().any().any(),
    'holiday years are 2016-2018': set(pd.to_datetime(holiday_result['date']).dt.year) == {2016, 2017, 2018},
}
check_df = pd.DataFrame([{'check': name, 'status': 'PASS' if passed else 'FAIL'} for name, passed in checks.items()])
display(check_df)
assert all(checks.values()), 'A02 validation failed; inspect the table above.'
print('FINAL STATUS: PASS')

,check,status
0,9 Olist CSV files,PASS
1,all Olist schemas pass,PASS
2,comparison output exists,PASS
3,holiday output exists,PASS
4,holiday dates unique,PASS
5,holiday required fields complete,PASS
6,holiday years are 2016-2018,PASS


FINAL STATUS: PASS
